# Notebook 03 — FF3 Time-Series Replication

**Milestone**: Jun 7, 2026 — validate FF3 against Fama & French (1993) Table 3  
**Guide**: `notes/ff3_replication_guide.md` — read that first!  
**Paper**: `papers/fama_french_1993.pdf` — have Table 3 open beside this notebook

### What you're doing
Running OLS: `Rp - Rf = α + β·MktRF + s·SMB + h·HML + ε`  
for each of 25 size/BM-sorted portfolios, then comparing results to Table 3 of the paper.

### What success looks like
- Mean |α| across 25 portfolios < 0.25 %/month
- Mean R² > 0.85
- SMB loadings decrease monotonically with SIZE
- HML loadings increase monotonically with B/M
- Results saved to `results/ff3_replication.csv` and committed

In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.factors.fama_french import load_public_ff3

---
## Step 1 — Load and inspect data

**Before coding**: Write out the regression equation on paper. What are the four variables? What does each coefficient mean?

Files you have:
- `../data/pulls/ff3_monthly.parquet` — factors (already accessible via `load_public_ff3()`)
- `../data/pulls/25_portfolios_5x5_vw.parquet` — 25 portfolio VW returns, monthly %

In [2]:
# Exercise 1.1 — Load both datasets
# ff3: use load_public_ff3() from src.factors.fama_french
# portfolios: pd.read_parquet('...')
# Print: shape, date range (min/max), first 3 rows for each

# YOUR CODE HERE


In [3]:
# Exercise 1.2 — Column mapping
# Print the 25 portfolio column names.
# Then fill in the blanks:
#   s1_b1 = ME1 BM1 = ________ (size descriptor, B/M descriptor)
#   s1_b5 = ME1 BM5 = ________
#   s5_b1 = ME5 BM1 = ________
#   s5_b5 = ME5 BM5 = ________

# YOUR CODE HERE


In [4]:
# Exercise 1.3 — Subset to the original FF1993 sample period
# Original paper: July 1963 – December 1991 (should give exactly 342 months)
# Subset BOTH datasets to this period
# Confirm: len(ff3_orig) == 342

# YOUR CODE HERE


**Checkpoint 1**: Before proceeding — can you answer these from the data?
1. What is the mean monthly return of the Mkt-RF factor in this period?
2. What is the mean monthly return of portfolio s1_b1? s5_b5?
3. Do small-value stocks outperform small-growth stocks in this period? By how much per year?

In [5]:
# Answer checkpoint questions here

# YOUR CODE HERE


---
## Step 2 — Align and create excess returns

The dependent variable in FF3 is **excess** portfolio return: `Rp - Rf`

Think about why before coding: the factors (Mkt-RF, SMB, HML) are themselves expressed as excess/spread returns. If you regress raw Rp on them, the intercept will absorb the risk-free rate and be misleadingly large.

In [6]:
# Exercise 2.1 — Create an aligned DataFrame
# Merge ff3_orig and portfolios_orig on the date index
# Check for NaNs: assert aligned.isnull().sum().sum() == 0
# (If there are NaNs, your date alignment has a bug — investigate before proceeding)

# YOUR CODE HERE


In [7]:
# Exercise 2.2 — Create excess returns for all 25 portfolios
# For each portfolio column: excess = portfolio_return - rf
# Easiest pattern: excess_df = portfolios_df.subtract(ff3['rf'], axis=0)
# Store in excess_df, shape should be (342, 25)

# YOUR CODE HERE


In [8]:
# Exercise 2.3 — Quick visual sanity check
# Plot cumulative returns (not excess) of s1_b1 vs s5_b5 over the sample period
# Do you see the value premium (s?_b5 > s?_b1) and size premium (s1 > s5)?

# YOUR CODE HERE


---
## Step 3 — OLS for ONE portfolio first

Always write the single-case version before the loop. It lets you:
1. Read the `model.summary()` and understand what you're extracting
2. Catch bugs early (wrong sample, wrong dependent variable, missing constant)
3. Compare manually to the paper before automating

In [9]:
# Exercise 3.1 — OLS for s1_b1 (Small Growth)
#
# Pattern:
#   X = sm.add_constant(factors[['mktrf', 'smb', 'hml']])
#   y = excess_df['s1_b1']
#   model = sm.OLS(y, X).fit()
#   print(model.summary())
#
# Do this, then identify in the output: alpha (const), beta, s, h, R-squared

# YOUR CODE HERE


In [10]:
# Exercise 3.2 — Read off your results for s1_b1 and compare to the paper
# Fill in this table manually:
#
#   alpha   = ____  (paper Table 3, ME1 BM1 row)
#   t(alpha)= ____
#   beta    = ____
#   s       = ____
#   h       = ____
#   R²      = ____
#
# Are the signs what you expected? If h is positive for s1_b1, something is wrong.

print("s1_b1 (Small Growth) — my results vs paper:")
# YOUR CODE HERE to extract and print the 6 numbers


s1_b1 (Small Growth) — my results vs paper:


In [11]:
# Exercise 3.3 — Repeat for s5_b5 (Large Value)
# Expected: positive h (value tilt), negative s (large cap), beta close to 1

# YOUR CODE HERE


---
## Step 4 — Loop over all 25 portfolios

Now write the function and loop. The function should be clean enough to go into `src/models/linear.py` later.

**Design principle**: the function takes arrays, not column names. It should work for any OLS, not just FF3.

In [12]:
# Exercise 4.1 — Write the function
#
# def run_ts_ols(y: pd.Series, X: pd.DataFrame) -> dict:
#     """
#     Run time-series OLS with a constant. Return dict of key statistics.
#     
#     Returns: alpha, t_alpha, beta, s, h, r2, n
#     (beta = mktrf coef, s = smb coef, h = hml coef)
#     """
#     ... your implementation ...

# Hints:
#   - model.params['const'] gives alpha
#   - model.tvalues['const'] gives t(alpha)
#   - model.rsquared gives R²
#   - model.nobs gives N

# YOUR CODE HERE


In [13]:
# Exercise 4.2 — Run for all 25 portfolios
#
# factors_X = sm.add_constant(aligned[['mktrf', 'smb', 'hml']])
#
# results = []
# for col in [list of 25 portfolio columns]:
#     y = excess_df[col]
#     r = run_ts_ols(y, factors_X)
#     r['portfolio'] = col
#     results.append(r)
#
# results_df = pd.DataFrame(results).set_index('portfolio')
# print(results_df.round(3))

# YOUR CODE HERE


In [14]:
# Exercise 4.3 — Validate the monotone patterns
#
# Check 1: SMB loading (s) decreases with SIZE
#   For each SIZE quintile (s1..s5), compute mean 's' loading across all B/M groups
#   Result should be: s1 > s2 > s3 > s4 > s5
#
# Check 2: HML loading (h) increases with B/M
#   For each B/M quintile (b1..b5), compute mean 'h' loading across all SIZE groups
#   Result should be: b1 < b2 < b3 < b4 < b5
#
# If either pattern is wrong: go back and check your excess return calculation.

# YOUR CODE HERE


In [15]:
# Exercise 4.4 — Summary statistics
#
# Print:
#   Mean |alpha|  across 25 portfolios (target: < 0.25 %/month)
#   Max  |alpha|  (which portfolio has it?)
#   Mean R²       (target: > 0.85)
#   Number of portfolios with |t(alpha)| > 2

# YOUR CODE HERE


---
## Step 5 — Build the Table 3 equivalent and save

The 5×5 alpha matrix is the canonical output — it's what appears in papers when authors present FF3 validation.

In [16]:
# Exercise 5.1 — Alpha matrix (5×5)
#
# The portfolio columns are ordered: s1_b1, s1_b2, ..., s1_b5, s2_b1, ..., s5_b5
# So results_df['alpha'].values.reshape(5, 5) gives you the matrix directly.
#
# Create alpha_df with:
#   index = ['Small', '2', '3', '4', 'Large']   (size quintiles)
#   columns = ['Growth', '2', '3', '4', 'Value'] (B/M quintiles)

# YOUR CODE HERE

# Then print it rounded to 2 decimals — your Table 3 equivalent


In [17]:
# Exercise 5.2 — Save full results
#
# Save results_df to: ../results/ff3_replication.csv
# Required columns: portfolio, alpha, t_alpha, beta, s, h, r2, n
#
# Also add columns: size_q (1-5) and bm_q (1-5) parsed from the portfolio name
# This makes the CSV self-contained for downstream use

# YOUR CODE HERE


In [18]:
# Exercise 5.3 — Print validation summary
#
# Print a box like this:
#
#   === FF3 Replication Validation (Jul 1963 – Dec 1991, N=342) ===
#   Mean |alpha|:  0.XX %/month   [target: < 0.25]
#   Max  |alpha|:  0.XX %/month   (portfolio: ____)
#   Mean R²:       0.XX            [target: > 0.85]
#   Portfolios with |t(α)| > 2: X / 25
#   Results saved: results/ff3_replication.csv
#   ================================================================

# YOUR CODE HERE


---
## Paper comparison — Table 3 validation

**Before committing**: fill in this comparison table from the paper's Table 3. Your numbers may differ slightly (French revises historical data), but should be within ±0.05% for alpha and ±0.02 for R².

In [19]:
# Build a 4-row comparison table: 4 corners of the 5x5 grid
# Manual lookup from paper + your computed values

comparison = pd.DataFrame([
    # Fill in paper_alpha, paper_r2 from Table 3 after reading the paper
    {'portfolio': 's1_b1 (Small Growth)', 'paper_alpha': None, 'my_alpha': None, 'paper_r2': None, 'my_r2': None},
    {'portfolio': 's1_b5 (Small Value)',  'paper_alpha': None, 'my_alpha': None, 'paper_r2': None, 'my_r2': None},
    {'portfolio': 's5_b1 (Large Growth)', 'paper_alpha': None, 'my_alpha': None, 'paper_r2': None, 'my_r2': None},
    {'portfolio': 's5_b5 (Large Value)',  'paper_alpha': None, 'my_alpha': None, 'paper_r2': None, 'my_r2': None},
])

# YOUR CODE HERE: fill in my_alpha and my_r2 from results_df
# Then fill in paper_alpha and paper_r2 by reading Table 3

print(comparison.to_string(index=False))

           portfolio paper_alpha my_alpha paper_r2 my_r2
s1_b1 (Small Growth)        None     None     None  None
 s1_b5 (Small Value)        None     None     None  None
s5_b1 (Large Growth)        None     None     None  None
 s5_b5 (Large Value)        None     None     None  None


---
## Overachieve — Extensions

If you finish the core before 10:45 AM, try one of these.

In [20]:
# Extension A — Alpha heatmap
# Use seaborn.heatmap to visualize the 5x5 alpha matrix
# Use cmap='RdBu_r', center=0, annot=True, fmt='.2f'
# Title: 'FF3 alpha (% / month), Jul 1963 – Dec 1991'
# Save to: ../figures/ff3_alpha_heatmap.png

# YOUR CODE HERE


In [21]:
# Extension B — t-stat heatmap
# Same but use t_alpha. Mark cells where |t| > 2 with a border or annotation.
# Which portfolios still have statistically significant alphas after FF3?

# YOUR CODE HERE


In [22]:
# Extension C — Newey-West standard errors
# Re-run OLS for all 25 portfolios using cov_type='HAC', maxlags=3
# Create a new column t_alpha_nw in results_df
# Do any portfolios change significance (cross the |t|=2 threshold)?
# This is the correct standard error in the literature

# YOUR CODE HERE


In [23]:
# Extension D — CAPM comparison (1-factor vs 3-factor)
# For each portfolio, also run: Rp - Rf = alpha_capm + beta * MktRF + e
# Store alpha_capm in results_df
# Then compute: alpha_reduction = (alpha_capm - alpha_ff3) / alpha_capm
#   = how much of CAPM's unexplained return does FF3 explain?
# This is the lead-in to the Fama-MacBeth regression (next RFZ task)

# YOUR CODE HERE


---
## Commit checklist

```bash
cd ~/Code/RegimeFactorZoo
git status
# Should see: notebooks/03_ff3_replication.ipynb, results/ff3_replication.csv
# Optionally: figures/ff3_alpha_heatmap.png

git add notebooks/03_ff3_replication.ipynb \
        results/ff3_replication.csv \
        data/pulls/25_portfolios_5x5_vw.parquet \
        notes/ff3_replication_guide.md

git commit -m "feat: FF3 baseline replication validated"
```

**Milestone passes when**:
- `results/ff3_replication.csv` exists and has 25 rows
- Mean |alpha| < 0.25 %/month
- Mean R² > 0.85
- Commit is in git log